# 4 Your own series, your own negative result

Notebooks 01-03 ran the toolkit on series somebody else picked: daily sunspot counts
and solar power. This notebook is where you pick your own, from the same archive, and
run the lesson's toolkit on it yourself:

1. establish the naive/seasonal-naive/linear baseline (`02_basemodels`' habit) --
   **before** you touch anything else;
2. read the spectrum, decompose it, and try N-BEATS
   (`03_decomposition_and_nbeats`);
3. judge whether a point forecast is even the right output for this series.

**What you bring to the lesson is your runs, discussed there in mlflow, and they must
include at least one negative result you produced and are willing to stand behind** — same requirement as
`1_pytorch_intro/05_exercises`. "We tried it, the baseline won, here is the number" is
a complete and respectable finding: a hypothesis that was tested and did not pay is a run
worth bringing. A set of runs that only shows things working has not done the assignment.

Keep the series you pick here. Lesson 3 searches architectures on it and lesson 6 puts
a distribution on its forecasts, and both are easier when you already know how it
behaves.

The `Monash-University/monash_tsf` configs used so far do not repeat: `sunspot`,
`solar_10_minutes`. Pick a different one — `traffic_hourly`, `nn5_daily`,
`kaggle_web_traffic_weekly`, `covid_deaths`, `us_births`, `saugeenday` and more all
load the same way (§4.1 below is the pattern, worked on `traffic_hourly` so you have a
template that runs end to end).

## 4.1 The pattern, worked once

`traffic_hourly` — San Francisco Bay Area road occupancy, hourly, 862 sensors, values
in [0, 1]. Genuinely noisier than anything else in this lesson (`evidence.md` §5
measured its spectral top-5/total at 0.38, the lowest of every series checked) —
picked deliberately so the worked example is not another easy win.

In [ ]:
import numpy as np
import torch
from pathlib import Path
from datasets import load_dataset
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from vectormesh.components.metrics import MAE
from vectormesh.training import Trainer, TrainerSettings

from windowing import window_index
from notebooktester import param

W, H = 168, 24
EPOCHS = param(10, test=1)
SEEDS = (0, 1, 2)


def load_series(
    config: str, n_series: int = 4, min_len: int = 6 * (W + H)
) -> list[torch.Tensor]:
    # the same loader shape 03_decomposition_and_nbeats uses --
    # swap `config` for any Monash-University/monash_tsf name and nothing else changes.
    ds = load_dataset(
        "Monash-University/monash_tsf",
        revision="refs/convert/parquet",
        data_files={"test": f"{config}/test/0000.parquet"},
    )["test"]
    out = []
    for row in ds:
        s = np.asarray(row["target"], dtype="float32")
        s = s[~np.isnan(s)]
        if len(s) >= min_len:
            out.append(torch.tensor(s))
        if len(out) >= n_series:
            break
    return out


def split_scale_window(series: list[torch.Tensor], stride: int = 6) -> dict:
    parts = {"train": [[], []], "val": [[], []], "test": [[], []]}
    for s in series:
        n = len(s)
        cuts = {
            "train": (0, int(0.6 * n)),
            "val": (int(0.6 * n), int(0.8 * n)),
            "test": (int(0.8 * n), n),
        }
        mu = s[: cuts["train"][1]].mean()
        sigma = s[: cuts["train"][1]].std().clamp(min=1e-6)
        for split, (a, b) in cuts.items():
            v = (s[a:b] - mu) / sigma
            win = v[window_index(len(v), W + H, stride)]
            parts[split][0].append(win[:, :W])
            parts[split][1].append(win[:, W:])
    return {k: (torch.cat(v[0]), torch.cat(v[1])) for k, v in parts.items()}


data = split_scale_window(load_series("traffic_hourly"))
xtr, ytr = data["train"]
xva, yva = data["val"]
print(f"train {tuple(xtr.shape)}   val {tuple(xva.shape)}")

In [ ]:
def mae(y: torch.Tensor, yhat: torch.Tensor) -> float:
    return float((y - yhat).abs().mean())


mase_scale = float((xtr[:, 1:] - xtr[:, :-1]).abs().mean())  # 02_basemodels 2.2
naive = xva[:, -1:].expand(-1, H)
seasonal = xva[:, -24:][:, :H]

print(
    f"{'naive (repeat last)':22s} MAE {mae(yva, naive):.4f}   MASE {mae(yva, naive) / mase_scale:.3f}"
)
print(
    f"{'seasonal naive (24h)':22s} MAE {mae(yva, seasonal):.4f}   MASE {mae(yva, seasonal) / mase_scale:.3f}"
)

# one batch covering the whole val split -- fit() shares this loader across seeds
validloader = DataLoader(TensorDataset(xva, yva), batch_size=len(xva))


def fit(make_model, seeds=SEEDS, epochs=EPOCHS, batch_size=64, lr=1e-3):
    # vectormesh.training.Trainer, once per seed -- see 02_basemodels for the same pattern.
    scores = []
    for seed in seeds:
        torch.manual_seed(seed)
        model = make_model()
        trainloader = DataLoader(
            TensorDataset(xtr, ytr), batch_size=batch_size, shuffle=True
        )
        settings = TrainerSettings(
            epochs=epochs,
            metrics=[MAE()],
            logdir=Path("modellogs").resolve(),
            train_steps=len(trainloader),
            valid_steps=len(validloader),
            optimizer_kwargs={"lr": lr},
            earlystop_kwargs=None,
        )
        trainer = Trainer(
            model=model,
            settings=settings,
            loss_fn=nn.MSELoss(),
            optimizer=torch.optim.Adam,
            traindataloader=trainloader,
            validdataloader=validloader,
            progress=False,
        )
        scores.append(trainer.loop().metric_dict["MAE"])
    return float(np.mean(scores)), float(np.std(scores))


class Linear(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc = nn.Linear(W, H)

    def forward(self, x):
        return self.fc(x)


lin_mae, lin_std = fit(Linear)
print(
    f"{'linear':22s} MAE {lin_mae:.4f} +/- {lin_std:.4f}   MASE {lin_mae / mase_scale:.3f}"
)

**Settle what you see before moving on.** On `traffic_hourly`, is the naive or the
seasonal-naive baseline stronger? Does the trained linear layer clear either of them by
more than its own seed spread? That answer is the first thing to read off your runs, and it
should be a number, not an impression — `02_basemodels` §2.1-2.2 is the reasoning to
reuse, not just the code.

## 4.2 Decompose it, try N-BEATS

`03_decomposition_and_nbeats` §3.2-3.5 has every class you need (`TrendBasis`,
`SeasonalityBasis`, `NBeatsBlock`, `NBeats`, `interpretable()`, `generic()`) — copy
them, do **not** re-derive them, and point `period=` at whatever `traffic_hourly`'s
spectrum (or your own series') actually shows, the way §3.1 read solar power's off an
FFT before assuming 24 was right.

Run the same three-way baseline table §3.4 does (linear / N-BEATS generic / N-BEATS
interpretable). Does the constrained architecture earn anything here, or does the
linear layer win again? Either answer is fine — §3.6 measured both directions on two
different series and that was the point, not an accident.

## 4.3 Is a point forecast the right output at all?

This one you answer in words and one plot, not with a model. Take 40 windows whose
histories are near-identical — nearest neighbours in window space — and plot what
actually happened next in each. If those 40 futures fan out widely, then **no single
number is the answer**, and the best point forecast in the world is still reporting the
middle of a distribution while saying nothing about its width. If the fan is narrow, a
point forecast is most of what there is to say.

State which one your series is, with the plot as evidence. That judgement is the entry
point to lesson 6, which builds the machinery — CRPS, Gaussian and quantile heads,
calibration — for the case where the fan is wide. Deciding *whether you need it* comes
first, and it is the cheaper skill.

## What you bring

Your runs, discussed in the lesson in mlflow, the way
`1_pytorch_intro/05_exercises` closes. Between them they cover:

1. Your series (name, config, why you picked it) and the naive/seasonal-naive/linear
   baseline table, **established first**.
2. The decomposition + N-BEATS result, with the period you chose and why.
3. Your neighbour-fan plot and the verdict it supports: does this series need a
   distribution, or is a point forecast honest here?
4. **At least one negative result** — a thing you tried that did not beat its baseline,
   run as carefully as the things that did. If everything you tried won, you have not
   tried enough things, or you are not being honest about the results. Either is worth noticing.

## Exercises

**E1 — pick deliberately, not randomly.** Before loading anything, read the Monash
config list (`evidence.md` §5 has several already characterised) and predict: does your
chosen series have a clean spectral peak like solar power, or a spread-out one like
traffic? Check your prediction against the data once you have loaded it.

**E2 — the MASE scale is not optional.** Whatever your final number is, name the MASE
denominator it was computed against, right beside it. A MASE with no stated
baseline is decoration, not a result — `02_basemodels` §2.2 says why.

**E3 — three seeds, always.** Every trained number you bring should come with a
+/- from at least 3 seeds. A single-seed win is an anecdote until you have checked it
survives re-running.

**E4 — carry it forward.** Keep the loader, the splits and the baseline table you built
here. `3_scaling_and_search/06_search` points a real ray search at exactly this shape of
problem, and it is far more interesting run on a series you already have an opinion
about than on one handed to you.